# Create Silver Layer

In [0]:
## Create silver schema

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.shop_flow_silver")

## Load Tables 

In [0]:
## Load Tables Dataframe

customers_df = spark.table(
    "workspace.shop_flow_bronze.customers"
)

orders_df = spark.table(
    "workspace.shop_flow_bronze.orders"
)

order_items_df = spark.table(
    "workspace.shop_flow_bronze.order_items"
)

products_df = spark.table(
    "workspace.shop_flow_bronze.products"
)

sellers_df = spark.table(
    "workspace.shop_flow_bronze.sellers"
)

order_payments_df = spark.table(
    "workspace.shop_flow_bronze.order_payments"
)

order_reviews_df = spark.table(
    "workspace.shop_flow_bronze.order_reviews"
)

product_categories_df = spark.table(
    "workspace.shop_flow_bronze.product_category_translation"
)

geolocation_df = spark.table(
    "workspace.shop_flow_bronze.geolocation"
)

## Init

In [0]:
from pyspark.sql.functions import (col, count, avg, sum, min, max, countDistinct, when, length, lpad, initcap, concat_ws, trim, to_date, to_timestamp)

# CUSTOMERS

In [0]:
## Convert the zip code 4 digits to 5 digits


customers_silver = customers_df.withColumn('customer_zip_code_prefix', lpad(col('customer_zip_code_prefix').cast("string"), 5, '0'))

display(customers_silver)


In [0]:
customers_silver = customers_silver.withColumn(
    'customer_city',
    initcap(col("customer_city"))
)

display(customers_silver)

In [0]:
# Save the changes into the silver table

customers_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.shop_flow_silver.customers")

In [0]:
customers_silver_df = spark.table("workspace.shop_flow_silver.customers")

In [0]:
customers_silver_df.display()

# ORDER_ITEMS

## Ingest to silver layer

In [0]:
order_items_df.write.mode("overwrite").saveAsTable("workspace.shop_flow_silver.order_items")

In [0]:
order_items_silver = spark.table("workspace.shop_flow_silver.order_items")

order_items_silver.display()

# ORDER_PAYMENTS

## Ingest to silver layer

In [0]:
order_payments_df.write.mode("overwrite").saveAsTable("workspace.shop_flow_silver.order_payments")
order_payments_silver = spark.table("workspace.shop_flow_silver.order_payments")

order_payments_silver.display()

# ORDER_REVIEWS

## Drop Nulls order_id

In [0]:
order_reviews_silver = order_reviews_df.filter(col("order_id").isNotNull())

order_reviews_silver.filter(col("order_id").isNull()).display()

In [0]:
order_reviews_silver = order_reviews_silver.withColumn(
    "review_order_key",
    concat_ws("_", col("review_id"), col("order_id"))
)

order_reviews_silver.display()


## Standarization review_score

In [0]:
order_reviews_silver = order_reviews_silver.withColumn(
    "review_score",
    when(
        trim(col("review_score")).isin("1", "2", "3", "4", "5"),
        trim(col("review_score")).cast("int")
    ).otherwise(None)

)
order_reviews_silver.display()

## Data Type Casting

In [0]:
order_reviews_silver = order_reviews_silver.withColumn(
    "review_creation_date",
    when(
        trim(col("review_creation_date")).rlike(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"),
        trim(col("review_creation_date").cast("date")
    )
).otherwise(None)
).withColumn(
    "review_answer_timestamp",
when(
        trim(col("review_answer_timestamp")).rlike(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"),
        trim(col("review_creation_date").cast("timestamp")
    )
).otherwise(None)
)

order_reviews_silver.display()


## Ingest to silver layer

In [0]:
order_reviews_silver.write.mode("overwrite").saveAsTable("workspace.shop_flow_silver.order_reviews")
order_reviews_silver = spark.table("workspace.shop_flow_silver.order_reviews")

order_reviews_silver.display()

# ORDERS

## Ingest to silver layer

In [0]:
orders_df.write.mode("overwrite").saveAsTable("workspace.shop_flow_silver.orders")
orders_silver = spark.table("workspace.shop_flow_silver.orders")
orders_silver.display()

# PRODUCTS

## Replcae the 0 value to nulls

In [0]:
products_silver = products_df.withColumn(
    "product_weight_g",
    when(
        col("product_weight_g") == 0 , None
    ).otherwise(col("product_weight_g"))
)

products_silver.display()

## Ingest to silver layer

In [0]:
products_silver.write.mode("overwrite").saveAsTable("workspace.shop_flow_silver.products")
products_silver = spark.table("workspace.shop_flow_silver.products")
products_silver.display()

# SELLERS

## Fix invalid zip codes


In [0]:
sellers_silver = sellers_df.withColumn(
    "seller_zip_code_prefix",
    lpad(col("seller_zip_code_prefix").cast("string"), 5, "0")
)

sellers_silver.filter(length(col("seller_zip_code_prefix")) != 5).display()

## Ingest to silver layer

In [0]:
sellers_silver.write.mode("overwrite").saveAsTable("workspace.shop_flow_silver.sellers")
sellers_silver = spark.table("workspace.shop_flow_silver.sellers")
sellers_silver.display()